# Project INTEGRATE — IPD Meta-Analysis
**Dr. Kari Northeim | UNT Health Science Center**

Replicates: Mun et al. (2022), *Prevention Science*

This notebook runs a two-step individual participant data (IPD) meta-analysis of brief alcohol interventions using the Project INTEGRATE dataset.

In [ ]:
# ── Cell 1: Install libraries (JupyterLite only) ──────────
import micropip
await micropip.install(['pandas', 'numpy', 'scipy', 'statsmodels'])
print('Libraries installed.')

In [ ]:
# ── Cell 2: Import libraries ──────────────────────────────
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
import io, warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

In [ ]:
# ── Cell 3: Load dataset from GitHub ─────────────────────
from pyodide.http import pyfetch

DATA_URL = 'https://raw.githubusercontent.com/drknortheim/Python_for_Addiction_Researchers/main/data/dat_all_PS.csv'

response = await pyfetch(DATA_URL)
content  = await response.string()
df = pd.read_csv(io.StringIO(content))

print('Dataset loaded successfully')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── Cell 4: Explore the data ──────────────────────────────
print('── Time points ──')
print(df['time'].value_counts().sort_index())

print('\n── Studies ──')
print(sorted(df['study'].unique()))

print('\n── Outcome variable (y = drinks/week) ──')
print(df['y'].describe())
print(f"Zero drinkers: {(df['y']==0).mean()*100:.1f}%")

In [ ]:
# ── Cell 5: Split by follow-up time point ────────────────
df_1mo  = df[df['time'] == 1].copy()
df_6mo  = df[df['time'] == 6].copy()
df_12mo = df[df['time'] == 12].copy()

print(f'1-3 months:  {len(df_1mo):,} participants')
print(f'6 months:    {len(df_6mo):,} participants')
print(f'9-12 months: {len(df_12mo):,} participants')

In [ ]:
# ── Cell 6: Helper functions ──────────────────────────────
INTERVENTION_VARS = ['int_GMI', 'int_MI_PF', 'int_PF']
COVARIATE_VARS    = ['m0', 'gender', 'year_new', 'race_new']
PROBLEM_STUDIES   = ['2', '7.1']

def is_zero_inflated(y_series, threshold=0.10):
    y = y_series.dropna()
    zero_rate_obs = (y == 0).mean()
    zero_rate_exp = np.exp(-y.mean())
    return zero_rate_obs > zero_rate_exp + threshold

def fit_study(study_df, study_id):
    covariates = COVARIATE_VARS.copy()
    if study_id in PROBLEM_STUDIES:
        covariates = [v for v in covariates if v != 'gender']
    int_vars = [v for v in INTERVENTION_VARS
                if v in study_df.columns and study_df[v].notna().any()]
    all_vars = [v for v in int_vars + covariates if study_df[v].notna().any()]
    model_df = study_df[all_vars + ['y']].dropna()
    if len(model_df) < 20:
        return None
    label = 'MZIP-candidate (using NB)' if is_zero_inflated(model_df['y']) else 'NB'
    print(f'  Study {study_id}: {label}, n={len(model_df)}')
    formula = 'y ~ ' + ' + '.join(all_vars)
    try:
        model  = smf.negativebinomial(formula, data=model_df).fit(disp=False)
        result = model.summary2().tables[1]
        int_rows = result[result.index.str.startswith('int_')]
        return int_rows[['Coef.', 'Std.Err.']].rename(
            columns={'Coef.': 'estimate', 'Std.Err.': 'se'})
    except Exception as e:
        print(f'  Study {study_id}: failed — {e}')
        return None

def random_effects_meta(estimates, ses):
    estimates = np.array(estimates); ses = np.array(ses)
    variances = ses**2; weights = 1/variances
    fe_est = np.sum(weights*estimates)/np.sum(weights)
    Q  = np.sum(weights*(estimates-fe_est)**2)
    k  = len(estimates); df = k-1
    c  = np.sum(weights) - np.sum(weights**2)/np.sum(weights)
    tau2 = max(0, (Q-df)/c)
    re_w   = 1/(variances+tau2)
    re_est = np.sum(re_w*estimates)/np.sum(re_w)
    re_se  = np.sqrt(1/np.sum(re_w))
    ci_lo  = re_est - 1.96*re_se; ci_hi = re_est + 1.96*re_se
    z      = re_est/re_se
    p      = 2*(1-stats.norm.cdf(abs(z)))
    i2     = max(0, (Q-df)/Q*100) if Q>0 else 0
    return {'estimate':re_est,'se':re_se,'ci_lower':ci_lo,'ci_upper':ci_hi,
            'z':z,'p_value':p,'tau2':tau2,'Q':Q,'I2':i2,'k':k}

def run_meta_analysis(subset_df, label):
    print(f'\n{"="*55}\n  {label}\n{"="*55}')
    all_results = []
    for study_id in sorted(subset_df['study'].unique()):
        result = fit_study(subset_df[subset_df['study']==study_id].copy(), study_id)
        if result is not None:
            result['study'] = study_id
            all_results.append(result)
    if not all_results:
        print('No results.'); return None
    combined = pd.concat(all_results)
    print(f'\n{"Intervention":<15} {"Est":>7} {"SE":>7} {"95% CI":>20} {"p":>8} {"I2":>7} {"k":>4}')
    print('-'*65)
    meta_summary = {}
    for iv in INTERVENTION_VARS:
        rows = combined[combined.index.str.startswith(iv)]
        if len(rows) < 2: continue
        m = random_effects_meta(rows['estimate'].values, rows['se'].values)
        meta_summary[iv] = m
        ci = f'[{m["ci_lower"]:+.3f}, {m["ci_upper"]:+.3f}]'
        print(f'{iv:<15} {m["estimate"]:>+7.3f} {m["se"]:>7.3f} {ci:>20} {m["p_value"]:>8.4f} {m["I2"]:>6.1f}% {m["k"]:>4}')
    return meta_summary

print('Helper functions ready.')

In [ ]:
# ── Cell 7: Run meta-analysis — all three time points ─────
results_1mo  = run_meta_analysis(df_1mo,  '1-3 Months Follow-Up')
results_6mo  = run_meta_analysis(df_6mo,  '6 Months Follow-Up')
results_12mo = run_meta_analysis(df_12mo, '9-12 Months Follow-Up')

In [ ]:
# ── Cell 8: Export results to a summary table ─────────────
rows = []
for label, results in [('1-3mo', results_1mo), ('6mo', results_6mo), ('12mo', results_12mo)]:
    if results is None: continue
    for iv, m in results.items():
        rows.append({'timepoint': label, 'intervention': iv,
                     'estimate': round(m['estimate'],4), 'se': round(m['se'],4),
                     'ci_lower': round(m['ci_lower'],4), 'ci_upper': round(m['ci_upper'],4),
                     'p_value':  round(m['p_value'],4),  'I2': round(m['I2'],1),
                     'k_studies': m['k']})

results_df = pd.DataFrame(rows)
print('\n── Summary Table ──')
results_df